# 140 — Robots colaborativos y seguridad física

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("safety", seed=140)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


## Solución 1 — S con dos sensores

Con `t_r = 0.15`: `S = 1.6·0.35 + 0.5·0.15 + 0.05 + 0.25 = 0.56 + 0.075 +
0.05 + 0.25 = 0.935 m`.
Con `t_r = 0.05`: `S = 1.6·0.25 + 0.5·0.05 + 0.05 + 0.25 = 0.40 + 0.025 +
0.05 + 0.25 = 0.725 m`.
El sensor rápido compra **21 cm** de cercanía. Nota que el término dominante
es `v_h·(t_r + t_s)`: el humano recorre distancia mientras el sistema
reacciona y frena — por eso sensado rápido y frenos rápidos valen tanto como
un robot lento.


In [ ]:
v_h, v_r, t_s, C = 1.6, 0.5, 0.2, 0.25
for t_r in (0.15, 0.05):
    S = v_h*(t_r+t_s) + v_r*t_r + v_r*t_s/2 + C
    print(f"t_r={t_r}: S={S:.3f} m")


## Solución 2 — Modos

- (a) **SSM**: movimiento simultáneo sin contacto previsto; la velocidad se
  regula con la distancia.
- (b) **Guiado manual**: es la definición del modo (y la base de la
  programación por demostración de la clase 138).
- (c) **PFL**: hay contacto esperado; exige diseño biomecánico del contacto
  (masa efectiva, superficies, umbrales por región corporal).
- (d) **Parada monitorizada de seguridad**: entradas esporádicas; detener y
  reanudar es más simple y productivo que instrumentar SSM continuo.


## Solución 3 — Curva v_max(d)

La curva resultante es creciente con d: a 0.5 m solo permite ~0.1-0.2 m/s, a
0.8 m del orden de 0.5-0.6, a 1.1 m ~1.0 (consistente con el README) y a
1.5 m ya admite el máximo del robot. La forma importa más que los decimales:
es un **lazo de regulación velocidad-distancia**, no un umbral binario
cerca/lejos — así el robot colabora de forma fluida en lugar de alternar
marcha/parada.


In [ ]:
def S(v):
    t_r, v_h, C = 0.1, 1.6, 0.2
    t_s = 0.3 * v / 1.0
    return v_h*(t_r+t_s) + v*t_r + v*t_s/2 + C

def v_max(d):
    best = 0.0
    v = 0.0
    while v <= 1.5:
        if S(v) <= d:
            best = v
        v = round(v + 0.01, 2)
    return best

for d in (0.5, 0.8, 1.1, 1.5):
    print(f"d={d} m -> v_max={v_max(d):.2f} m/s")


## Solución 4 — Tabla de traducción física → digital

| Seguridad física | Análogo en agente de computer use |
|---|---|
| Parada de emergencia | Kill-switch humano que aborta la sesión del agente al instante |
| SSM (velocidad ∝ distancia) | Autonomía ∝ reversibilidad: acciones libres si son deshacibles, más lentas/supervisadas cuanto más cerca de lo irreversible |
| PFL (contacto bajo umbral) | Acciones de bajo impacto permitidas sin confirmación (leer, scroll); límite duro en magnitud (n.º de archivos, importe máximo) |
| Evaluación de riesgos de la aplicación | Revisar la tarea completa (permisos, credenciales, datos accesibles), no solo el modelo — el mismo agente es seguro o no según qué "herramienta sujeta" |

El JSON del lab muestra la misma estructura: decisiones observables +
`limitations` que declaran lo que la demo no garantiza.


In [ ]:
from ai_evolution.labs import run_lab

result = run_lab("safety", seed=140)
assert result["kind"] == "safety"
print(result["evidence"])
print(result["limitations"])
